# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [21]:
!pip install spacy --quiet
!python -m spacy download es_core_news_lg --quiet
import es_core_news_lg
import spacy
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


nlp = spacy.load("es_core_news_lg")
npl = es_core_news_lg.load()
doc = nlp("El gato se sentó en la alfombra.")
print([(w.text, w.pos_) for w in doc])


def limpiar_texto(texto):
    doc = nlp.make_doc(texto.strip().lower())  
    conservar = {"no", "ni", "sin", "nunca", "tampoco"}

    palabras_limpias = [ 
        token.text
        for token in doc
        if not token.is_punct
        and not token.is_space
        and (not token.is_stop or token.text in conservar or token.like_num)
    ]

    return " ".join(palabras_limpias)  # Unimos las palabras para vectorizar el texto.

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
[('El', 'DET'), ('gato', 'NOUN'), ('se', 'PRON'), ('sentó', 'VERB'), ('en', 'ADP'), ('la', 'DET'), ('alfombra', 'NOUN'), ('.', 'PUNCT')]


## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





<div style="color: #172554; border: 1px solid #93c5fd; border-left: 7px solid #2563eb; border-radius: 10px; padding: 20px 24px; background: #eff6ff; margin: 12px 0; line-height: 1.65;">
  <h2 style="color: #1e3a8a; margin-top: 0;">1.1. Aplicación elegida: asistente virtual para una clínica dermatológica</h2>
  <p>Para este trabajo se adapta y busca perfeccionar un chatbot desarrollado previamente para una clienta del área de dermatología, utilizando <strong>datos anonimizados</strong>. Se usa el nombre ficticio <strong>Clínica Piel Sana</strong>. El objetivo es responder consultas frecuentes de pacientes mediante respuestas almacenadas en el dataset.</p>
  <p>El asistente cubre temas habituales como <strong>saludos y cierres, horarios, ubicación y contacto, turnos, obras sociales, tratamientos, preparación y cuidados, precios, urgencias y derivaciones</strong>. El dataset también incluye ejemplos de consultas fuera de alcance y respuestas de derivación a recepción, aunque su recuperación correcta no está garantizada.</p>
  <p>Este caso de uso es adecuado para un chatbot de recuperación porque las preguntas suelen repetirse con distintas redacciones y se busca brindar respuestas consistentes, claras y alineadas con la información oficial de la clínica.</p>
</div>

<div style="color: #143322; border: 1px solid #86efac; border-left: 7px solid #16a34a; border-radius: 10px; padding: 20px 24px; background: #f0fdf4; margin: 12px 0; line-height: 1.65;">
  <h2 style="color: #166534; margin-top: 0;">1.2. Dataset de preguntas y respuestas</h2>
  <p>El dataset se almacenó en el archivo <code style="color: #143322; background: #dcfce7; padding: 2px 5px; border-radius: 4px;">dataset_1.csv</code> y contiene <strong>227 pares de pregunta–respuesta</strong>, por lo que supera ampliamente el mínimo de 20 elementos solicitado.</p>
  <p>Cada registro posee tres campos: <code style="color: #143322; background: #dcfce7; padding: 2px 5px; border-radius: 4px;">pregunta</code>, <code style="color: #143322; background: #dcfce7; padding: 2px 5px; border-radius: 4px;">respuesta</code> y <code style="color: #143322; background: #dcfce7; padding: 2px 5px; border-radius: 4px;">categoria</code>. Las categorías permiten organizar las consultas por tema y luego analizar en cuáles el chatbot obtiene mejores o peores resultados.</p>
  <p>A continuación se carga y visualiza el DataFrame completo, junto con la cantidad de ejemplos disponibles por categoría.</p>
</div>

In [22]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('dataset_1.csv')


df['pregunta_limpia'] = df['pregunta'].apply(limpiar_texto)


print(f'Cantidad total de pares pregunta–respuesta: {len(df)}')
display(df)

print('Cantidad de preguntas por categoría:')
display(df['categoria'].value_counts().rename_axis('categoria').reset_index(name='cantidad'))

print('Control de limpieza: pregunta original y procesada')
display(df[['pregunta', 'pregunta_limpia']].head(10))

Cantidad total de pares pregunta–respuesta: 227


,pregunta,respuesta,categoria,pregunta_limpia
0,hola,¡Hola! Soy el asistente virtual de Clínica Pie...,saludo_cierre,hola
1,buenas tardes,¡Hola! Soy el asistente virtual de Clínica Pie...,saludo_cierre,tardes
2,hola buen dia como estan,¡Hola! Soy el asistente virtual de Clínica Pie...,saludo_cierre,hola
3,holaa necesito info,¡Hola! Soy el asistente virtual de Clínica Pie...,saludo_cierre,holaa necesito info
4,"buenas, una consulta",¡Hola! Soy el asistente virtual de Clínica Pie...,saludo_cierre,consulta
...,...,...,...,...
222,puedo tomar ibuprofeno antes del tratamiento?,No damos indicaciones de medicamentos ni crema...,urgencias_derivacion,tomar ibuprofeno tratamiento
223,quien gano el partido?,Solo puedo ayudarte con consultas de la clínic...,fuera_de_alcance,gano partido
224,me recomendas una pizzeria,Solo puedo ayudarte con consultas de la clínic...,fuera_de_alcance,recomendas pizzeria
225,como va a estar el clima,Solo puedo ayudarte con consultas de la clínic...,fuera_de_alcance,clima


Cantidad de preguntas por categoría:


,categoria,cantidad
0,tratamientos,42
1,precios,39
2,preparacion_cuidados,32
3,turnos_portal,25
4,obras_sociales,21
5,urgencias_derivacion,20
6,horarios,17
7,ubicacion_contacto,14
8,saludo_cierre,13
9,fuera_de_alcance,4


Control de limpieza: pregunta original y procesada


,pregunta,pregunta_limpia
0,hola,hola
1,buenas tardes,tardes
2,hola buen dia como estan,hola
3,holaa necesito info,holaa necesito info
4,"buenas, una consulta",consulta
5,gracias,gracias
6,muchas gracias genia,gracias genia
7,listo gracias chau,listo gracias chau
8,"perfecto, gracias!!",perfecto gracias
9,sos un robot?,sos robot


<div style="color: #172554; background: #eff6ff; border: 1px solid #93c5fd; border-left: 7px solid #2563eb; border-radius: 10px; padding: 20px 24px; margin: 12px 0; line-height: 1.65;">
  <h2 style="color: #1e3a8a; margin-top: 0;">2. Chatbot con TF-IDF y similitud del coseno</h2>
  <p>Cada pregunta se convierte en un vector de pesos: TF-IDF destaca las palabras que ayudan a distinguirla del resto del dataset. La consulta del usuario se transforma con el mismo vocabulario y los mismos pesos aprendidos.</p>
  <p>La similitud del coseno permite comparar la consulta con todas las preguntas almacenadas. Se elige la de mayor similitud y se devuelve su respuesta asociada; el chatbot no genera una respuesta nueva.</p>
  <p>Esta versión usa las preguntas originales, con el procesamiento predeterminado de TF-IDF, sin filtrar stopwords ni lematizar.</p>
</div>

In [ ]:
vectorizer = TfidfVectorizer()
tdidf_matrix = vectorizer.fit_transform(df['pregunta'])  

def answer_question(user_question, df, tdidf_matrix, mostrar=True):
    user_question = user_question.strip()  
    if not user_question:
        return '¿Podés reformular la pregunta con más detalle?'
    user_q = vectorizer.transform([user_question])
    if user_q.nnz == 0:
        return 'No encontré términos conocidos. ¿Podés reformular la pregunta?'
    similarities = cosine_similarity(user_q, tdidf_matrix)
    most_similar_index = similarities.argmax()
    if mostrar: 
        print(f'Índice de la pregunta más similar: {most_similar_index}')
        print(f"Pregunta más similar: {df.iloc[most_similar_index]['pregunta']}")  
        print(f'Similitud: {similarities[0, most_similar_index]:.3f}') 
    return df.iloc[most_similar_index]['respuesta']

user_question = '¿Qué horarios tienen?'  
response = answer_question(user_question, df, tdidf_matrix)  
print(f'Chatbot: {response}') 

Índice de la pregunta más similar: 13
Pregunta más similar: hola q horarios tienen
Similitud: 0.834
Chatbot: Atendemos de lunes a viernes de 9 a 19 h y los sábados de 9 a 13 h. Domingos y feriados nacionales, cerrado.


<div style="color: #143322; background: #f0fdf4; border: 1px solid #86efac; border-left: 7px solid #16a34a; border-radius: 10px; padding: 20px 24px; margin: 12px 0; line-height: 1.65;">
  <h2 style="color: #166534; margin-top: 0;">3. Chatbot con embeddings de spaCy</h2>
  <p>Se utilizan los vectores preentrenados del modelo <strong>es_core_news_lg</strong>. Cada pregunta se representa mediante el promedio de los vectores de sus palabras, obteniendo un vector de 300 valores.</p>
  <p>Primero se aplica la limpieza: minúsculas y eliminación de puntuación, espacios y stopwords, conservando negaciones y números. No se usa lematización. La misma función se aplica a las consultas del usuario.</p>
  <p>Los vectores se reúnen en una matriz, con una pregunta por fila. Como en TF-IDF, se calcula la similitud del coseno y se devuelve la respuesta de la pregunta más parecida. Con esta limpieza, los aciertos pasaron de 11 a 19 sobre 40 consultas.</p>
</div>

In [ ]:

documentos = [nlp.make_doc(pregunta) for pregunta in df['pregunta_limpia']] 
embeddings_matrix = np.vstack([doc.vector for doc in documentos]) 
print(f'Dimensiones de la matriz: {embeddings_matrix.shape}') 

def answer_question_embeddings(user_question, df, embeddings_matrix, mostrar=True):  
    user_question = limpiar_texto(user_question)  
    if not user_question:  
        return 'Escribí una pregunta para que pueda ayudarte.'  
    user_doc = nlp.make_doc(user_question)  
    if user_doc.vector_norm == 0: 
        return 'No pude representar la consulta. ¿Podés reformularla?' 
    user_q = user_doc.vector.reshape(1, -1)  
    similarities = cosine_similarity(user_q, embeddings_matrix) 
    most_similar_index = similarities.argmax()  
    if mostrar:
        print(f"Pregunta más similar: {df.iloc[most_similar_index]['pregunta']}")  
        print(f'Similitud: {similarities[0, most_similar_index]:.3f}')  
    return df.iloc[most_similar_index]['respuesta']  

user_question = '¿Qué horarios tienen?'  
response = answer_question_embeddings(user_question, df, embeddings_matrix)  
print(f'Chatbot: {response}')  

Dimensiones de la matriz: (227, 300)
Pregunta más similar: atienden los feriados
Similitud: 0.638
Chatbot: En los feriados nacionales la clínica está cerrada. El portal de turnos y este chat siguen funcionando; lo que necesite a una persona lo responde recepción el siguiente día hábil.


<div style="color: #172554; background: #eff6ff; border: 1px solid #93c5fd; border-left: 7px solid #2563eb; border-radius: 10px; padding: 20px 24px; margin: 12px 0; line-height: 1.65;">
  <h2 style="color: #1e3a8a; margin-top: 0;">4. Prueba de funcionamiento de ambos chatbots</h2>
  <p>Se evalúan los dos chatbots con las mismas <strong>40 preguntas</strong> del archivo <code style="color: #172554; background: #dbeafe; padding: 2px 5px; border-radius: 4px;">preguntas_prueba.csv</code>. La respuesta esperada se busca por su identificador en <code style="color: #172554; background: #dbeafe; padding: 2px 5px; border-radius: 4px;">respuestas.csv</code>. Estas preguntas de prueba no se incorporan a las matrices de búsqueda.</p>

  <p>Se considera un <strong>acierto</strong> cuando la respuesta obtenida coincide con la esperada, ignorando únicamente diferencias de espacios. Acertar la categoría no es suficiente. Como este criterio es estricto, una respuesta alternativa que también sea válida requiere revisión manual.</p>
  <p>La tabla permite comparar las preguntas, las respuestas esperadas y las respuestas de ambos modelos. <strong>True</strong> indica un acierto y <strong>False</strong> un error. Al final se muestran la cantidad y el porcentaje de aciertos de cada chatbot.</p>
</div>

In [20]:

pruebas = pd.read_csv("preguntas_prueba.csv")
catalogo = pd.read_csv("respuestas.csv").set_index("respuesta_id")
filas = []


def normalizar(texto):
    return " ".join(texto.split())

for _, caso in pruebas.iterrows():
    pregunta = caso["pregunta"]
    esperada = catalogo.loc[caso["respuesta_esperada_id"], "respuesta"]

   
    tfidf = answer_question(pregunta, df, tdidf_matrix, mostrar=False)
    spacy = answer_question_embeddings(pregunta, df, embeddings_matrix, mostrar=False)

    filas.append({
        "pregunta": pregunta,
        "respuesta_esperada": esperada,
        "respuesta_tfidf": tfidf,
        "respuesta_spacy": spacy,
        "acierto_tfidf": normalizar(tfidf) == normalizar(esperada),
        "acierto_spacy": normalizar(spacy) == normalizar(esperada),
    })

resultados_pruebas = pd.DataFrame(filas)
display(resultados_pruebas)


total = len(resultados_pruebas)
aciertos_tfidf = resultados_pruebas["acierto_tfidf"].sum()
aciertos_spacy = resultados_pruebas["acierto_spacy"].sum()
print(f"TF-IDF: {aciertos_tfidf}/{total} aciertos ({aciertos_tfidf / total:.1%})")
print(f"spaCy: {aciertos_spacy}/{total} aciertos ({aciertos_spacy / total:.1%})")


,pregunta,respuesta_esperada,respuesta_tfidf,respuesta_spacy,acierto_tfidf,acierto_spacy
0,bnas tardes!,¡Hola! Soy el asistente virtual de Clínica Pie...,¡Hola! Soy el asistente virtual de Clínica Pie...,¡Hola! Soy el asistente virtual de Clínica Pie...,True,True
1,"mil gracias, re amables","¡De nada! Si necesitás algo más, escribinos po...","¡De nada! Si necesitás algo más, escribinos po...","¡De nada! Si necesitás algo más, escribinos po...",True,True
2,hasta q hora estan abiertos hoy,Atendemos de lunes a viernes de 9 a 19 h y los...,Atendemos de lunes a viernes de 9 a 19 h y los...,Atendemos de lunes a viernes de 9 a 19 h y los...,True,True
3,los domingos atienden?,"Los sábados atendemos de 9 a 13 h, solo con tu...","Los sábados atendemos de 9 a 13 h, solo con tu...",En los feriados nacionales la clínica está cer...,True,False
4,el 25 de mayo abren?,En los feriados nacionales la clínica está cer...,"Los sábados atendemos de 9 a 13 h, solo con tu...",En los feriados nacionales la clínica está cer...,False,True
5,escribi tarde y nadie me contesto,Este chat responde las 24 horas. Si tu consult...,"Para cambiar el día o el horario, abrí el link...",Este chat responde las 24 horas. Si tu consult...,False,True
6,me pasas la direcion,"Estamos en el barrio Centro, a dos cuadras de ...",El valor de otros tratamientos depende de la e...,"Listo, ya le avisé a recepción. Te van a respo...",False,False
7,hay lugar para estacionar cerca?,"No tenemos estacionamiento propio, pero hay un...","No tenemos estacionamiento propio, pero hay un...","No tenemos estacionamiento propio, pero hay un...",True,True
8,cual es su numero de telefono,Este WhatsApp es el canal principal de contact...,Atendemos de lunes a viernes de 9 a 19 h y los...,Este WhatsApp es el canal principal de contact...,False,True
9,kiero turno para la dermatologa,Podés reservar tu turno online en nuestro port...,El láser fraccionado hace microcolumnas de cal...,"Para cancelar, abrí el link del mail o WhatsAp...",False,False


TF-IDF: 19/40 aciertos (47.5%)
spaCy: 19/40 aciertos (47.5%)


## Conclusiones
### Pruebas de preprocesamiento

En las 40 consultas evaluadas, con coincidencia de respuesta esperada (ignorando espacios), se obtuvieron estos resultados:

| Configuración | TF-IDF | spaCy |
|---|---:|---:|
| Sin limpieza adicional de stopwords | 19/40 (47,5 %) | 11/40 (27,5 %) |
| Con limpieza, conservando negaciones y números, sin lematizar | 14/40 (35 %) | 19/40 (47,5 %) |

Se decidió usar **TF-IDF sin filtrado adicional y spaCy con limpieza**. Quitar stopwords no mejora siempre los resultados: puede eliminar coincidencias útiles en consultas cortas. TF-IDF ya reduce mediante IDF el peso de los términos presentes en muchos documentos. En el promedio de vectores de spaCy, filtrar palabras comunes puede ayudar a destacar términos relevantes. Estas son explicaciones posibles de los resultados observados.



### Alcance de los embeddings

es_core_news_lg contiene vectores preentrenados para vocabulario general en español; no es un diccionario de definiciones ni un modelo especializado en las intenciones de esta clínica. Representamos cada pregunta con el promedio de los vectores de sus tokens. Esa representación puede perder diferencias entre consultas, y términos específicos pueden no tener vector (por ejemplo, se observó ese problema con OSDE en el modelo utilizado). Una similitud alta no garantiza una respuesta correcta.

Referencia: [Vectores y similitud en spaCy](https://spacy.io/usage/linguistic-features#vectors-similarity).


### Comparación de errores concretos

Con la configuración final, ambos bots aciertan 19 de 40 respuestas (47,5 %), pero no las mismas: 14 consultas son correctas para ambos, 5 solo para TF-IDF y 5 solo para spaCy. Los dos fallan en las otras 16 consultas. Por eso, el porcentaje global no describe todas sus diferencias.

- **“al final no puedo ir, lo doy de baja”**: TF-IDF recupera la cancelación de turnos (TUR02), mientras spaCy devuelve la respuesta de urgencias (URG01). Es una confusión de intención.
- **“precio meso corporal”**: spaCy devuelve el precio de mesoterapia corporal (PRC05), mientras TF-IDF devuelve sus cuidados posteriores (PRE10). Identificar el tratamiento no alcanza: también hay que reconocer qué se pregunta sobre él.
- **“aceptan osde?”**: ambos devuelven medios de pago (PRC11), en vez de la respuesta esperada sobre obras sociales (OBR01). Muestra una limitación frente a una cobertura no representada en las preguntas del dataset.

### Limitaciones y posibles mejoras

La limpieza también elimina información útil: deja vacías dos preguntas del dataset, “donde estan?” y “es mi primera vez, que tengo que hacer”. Esto muestra que las stopwords deben revisarse según el tipo de consulta.

No se usa un umbral mínimo de similitud. Salvo entradas vacías o sin representación utilizable, los bots eligen la pregunta más cercana aunque no sea pertinente. Como mejora, se podría evaluar un umbral para pedir una reformulación cuando la coincidencia sea débil, sin asumir que una similitud alta garantiza una respuesta correcta.

Las configuraciones se eligieron después de observar estas mismas 40 preguntas. Por eso, los resultados describen estas pruebas y no garantizan el mismo rendimiento en consultas nuevas. Para comprobarlo, sería necesario evaluar otro conjunto de preguntas que no se haya utilizado para decidir los cambios.

## Fuentes y material consultado

**Material de clase — Procesamiento del Habla**

- [Intro_Embeddings(Google Colab)](https://colab.research.google.com/drive/1U5FxDkGP2KmNYhp7n0HXYl8xXlcfUesF?usp=sharing).
- [Chatbot-retrieval(Google Colab)](https://colab.research.google.com/drive/1mbVJrLCoLgc9XsQzrMX_GeraURbWEjx4?usp=sharing).

**Trabajo previo propio**

- Sánchez Varela, Agustín. [TP1: web scraping y preprocesamiento de texto](https://github.com/agus476/ProcesamientoDelHabla/blob/main/tp%201/SANCHEZ_AGUSTIN_PH_TP1.ipynb). Se retomó la estructura de limpieza con spaCy y filtrado de tokens, adaptándola a consultas cortas y sin lematización en la versión final.

**Documentación oficial**

- scikit-learn. [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html): representación de textos mediante TF-IDF.
- scikit-learn. [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html): comparación de vectores mediante similitud del coseno.
- spaCy. [Modelos en español: es_core_news_lg](https://spacy.io/models/es#es_core_news_lg).
- spaCy. [Vectores de palabras y similitud](https://spacy.io/usage/linguistic-features#vectors-similarity): representación de textos mediante vectores y sus limitaciones.
